# AHS-KT × Junyi 全量运行 Notebook

这本 Notebook 用于把 `/root/autodl-tmp/ahs-kt/data/junyi` 里的 Junyi 原始数据跑完整 AHS-KT 流程：

1. 读取 `junyi_ProblemLog_original.csv` 与 `junyi_Exercise_table.csv`；
2. 按用户划分 `train / valid / test`；
3. 基于训练用户计算题目/知识点难度；
4. 基于训练用户拟合 AHS 行为聚类；
5. 保存全量 `.npz`、metadata 和 config；
6. 使用项目原生 `AHSKTModel + fit_and_evaluate` 训练并评估；
7. 额外输出 `F1`。

## 为什么之前只跑了 1000？

原来的 `run_junyi_ahskt.ipynb` 是 quickstart 版本，参数区写了：

```python
MAX_USERS = 1000
```

所以它只随机抽了 1000 个满足最少交互数的用户。这个 full 版本默认：

```python
MAX_USERS = None
```

也就是保留所有满足 `MIN_INTERACTIONS` 的用户。

## 注意

Junyi 原始日志约 2.5GB，交互量约 2592 万。全量构建和训练都比较耗时，也会占用较多内存。第一次建议先执行到“保存 `.npz`”确认数据构建成功，再继续训练。

In [1]:
from pathlib import Path
import gc
import json
import os
import random
import sys
import time

PROJECT_ROOT = Path('/root/autodl-tmp/ahs-kt')
SRC_ROOT = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data/junyi'
RAW_JUNYI_PATH = DATA_DIR / 'junyi_ProblemLog_original.csv'
JUNYI_EXERCISE_PATH = DATA_DIR / 'junyi_Exercise_table.csv'

assert PROJECT_ROOT.exists(), f'找不到项目目录: {PROJECT_ROOT}'
assert RAW_JUNYI_PATH.exists(), f'找不到 Junyi 日志文件: {RAW_JUNYI_PATH}'
assert JUNYI_EXERCISE_PATH.exists(), f'找不到 Junyi 练习元数据文件: {JUNYI_EXERCISE_PATH}'

os.chdir(PROJECT_ROOT)
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_JUNYI_PATH =', RAW_JUNYI_PATH)
print('JUNYI_EXERCISE_PATH =', JUNYI_EXERCISE_PATH)

PROJECT_ROOT = /root/autodl-tmp/ahs-kt
RAW_JUNYI_PATH = /root/autodl-tmp/ahs-kt/data/junyi/junyi_ProblemLog_original.csv
JUNYI_EXERCISE_PATH = /root/autodl-tmp/ahs-kt/data/junyi/junyi_Exercise_table.csv


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ahskt.config import load_config
from ahskt.data.assist2012 import build_sequence_records, records_to_bundle, save_bundle_npz, shuffle_records
from ahskt.data.dataset import load_bundle_from_config
from ahskt.models.ahs_kt import AHSKTModel
from ahskt.training.engine import fit_and_evaluate

print('TensorFlow version =', tf.__version__)
print('GPU devices =', tf.config.list_physical_devices('GPU'))
for gpu_device in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu_device, True)
    except RuntimeError:
        pass


libgomp: Invalid value for environment variable OMP_NUM_THREADS


TensorFlow version = 2.8.0
GPU devices = [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 参数区

默认是全量 Junyi：`MAX_USERS = None`。

如果你只是想先试跑，把它改成一个整数，例如 `MAX_USERS = 5000`。

In [3]:
SEED = 2026
MAX_USERS = 5000          # None = 全量满足 MIN_INTERACTIONS 的用户；整数 = 抽样用户数
MIN_INTERACTIONS = 5
SEQUENCE_LENGTH = 100
N_CLUSTERS = 4
CLUSTER_SAMPLE_SIZE = 300_000
CHUNKSIZE = 1_000_000

BATCH_SIZE = 256
EPOCHS = 10
LEARNING_RATE = 1e-3
PATIENCE = 3

TASK_NAME = 'ahskt_junyi_full'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/junyi_full_run'
CONFIG_PATH = PROJECT_ROOT / 'configs/ahskt_junyi_full.json'
TRAIN_NPZ_PATH = DATA_DIR / 'junyi_full_train_ahskt.npz'
VALID_NPZ_PATH = DATA_DIR / 'junyi_full_valid_ahskt.npz'
TEST_NPZ_PATH = DATA_DIR / 'junyi_full_test_ahskt.npz'
METADATA_PATH = DATA_DIR / 'junyi_full_metadata.json'

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('TASK_NAME =', TASK_NAME)
print('MAX_USERS =', MAX_USERS)
print('OUTPUT_DIR =', OUTPUT_DIR)

TASK_NAME = ahskt_junyi_full
MAX_USERS = 5000
OUTPUT_DIR = /root/autodl-tmp/ahs-kt/outputs/junyi_full_run


## 第一遍扫描：统计用户与 exercise

这里不直接把 2.5GB 的原始日志全读进内存，而是按 chunk 统计：

- 每个用户的交互次数；
- 日志中实际出现过的 exercise 名称。

In [4]:
scan_start = time.time()
user_counts = None
exercise_names = set()
total_rows = 0

for chunk_idx, chunk in enumerate(pd.read_csv(
    RAW_JUNYI_PATH,
    usecols=['user_id', 'exercise'],
    dtype={'user_id': 'int32', 'exercise': 'string'},
    chunksize=CHUNKSIZE,
    low_memory=False,
)):
    total_rows += len(chunk)
    counts = chunk['user_id'].value_counts(sort=False)
    if user_counts is None:
        user_counts = counts
    else:
        user_counts = user_counts.add(counts, fill_value=0)
    exercise_names.update(chunk['exercise'].dropna().astype(str).unique().tolist())
    if (chunk_idx + 1) % 5 == 0:
        print(f'已扫描 chunk {chunk_idx + 1}, rows={total_rows:,}, users={len(user_counts):,}, exercises={len(exercise_names):,}')

user_counts = user_counts.astype(np.int64)
eligible_users = user_counts[user_counts >= MIN_INTERACTIONS].index.to_numpy(dtype=np.int64)
rng = np.random.default_rng(SEED)

if MAX_USERS is None:
    chosen_users = np.sort(eligible_users.copy())
else:
    chosen_users = np.sort(rng.choice(eligible_users, size=min(MAX_USERS, len(eligible_users)), replace=False))

chosen_user_set = set(int(user_id) for user_id in chosen_users)

print('Junyi 总交互数 =', f'{int(total_rows):,}')
print('Junyi 总用户数 =', f'{int(len(user_counts)):,}')
print('满足最少交互数的用户数 =', f'{int(len(eligible_users)):,}')
print('本次实际使用用户数 =', f'{int(len(chosen_users)):,}')
print('日志中 exercise 数 =', f'{len(exercise_names):,}')
print('扫描耗时(秒) =', round(time.time() - scan_start, 2))

已扫描 chunk 5, rows=5,000,000, users=185,059, exercises=718
已扫描 chunk 10, rows=10,000,000, users=212,350, exercises=720
已扫描 chunk 15, rows=15,000,000, users=226,909, exercises=720
已扫描 chunk 20, rows=20,000,000, users=237,627, exercises=721
已扫描 chunk 25, rows=25,000,000, users=246,197, exercises=722
Junyi 总交互数 = 25,925,992
Junyi 总用户数 = 247,606
满足最少交互数的用户数 = 175,359
本次实际使用用户数 = 5,000
日志中 exercise 数 = 722
扫描耗时(秒) = 31.2


In [5]:
all_users = np.array(chosen_users, dtype=np.int64)
train_user_ids, test_user_ids = train_test_split(all_users, test_size=0.1, random_state=SEED)
train_user_ids, valid_user_ids = train_test_split(train_user_ids, test_size=1/9, random_state=SEED)

train_user_ids = np.asarray(train_user_ids, dtype=np.int64)
valid_user_ids = np.asarray(valid_user_ids, dtype=np.int64)
test_user_ids = np.asarray(test_user_ids, dtype=np.int64)
train_user_set = set(int(user_id) for user_id in train_user_ids)

print('train / valid / test 用户数 =', len(train_user_ids), len(valid_user_ids), len(test_user_ids))

train / valid / test 用户数 = 4000 500 500


## 建立 question / concept 映射

- `question` 使用 Junyi 的 `exercise`；
- `concept` 优先使用 `topic`，没有 `topic` 时退化到 `area`，再退化到 `exercise`。

In [6]:
exercise_df = pd.read_csv(JUNYI_EXERCISE_PATH, usecols=['name', 'topic', 'area'])
exercise_df['name'] = exercise_df['name'].astype(str)
exercise_df['concept_name'] = exercise_df['topic'].fillna(exercise_df['area']).fillna(exercise_df['name']).astype(str)
exercise_to_concept = dict(zip(exercise_df['name'], exercise_df['concept_name']))

# 以原始日志中实际出现的 exercise 为准，避免只依赖 exercise 表。
question_names = sorted(str(name) for name in exercise_names)
question_to_id = {name: idx + 1 for idx, name in enumerate(question_names)}

concept_names = sorted({exercise_to_concept.get(name, name) for name in question_names})
concept_to_id = {name: idx + 1 for idx, name in enumerate(concept_names)}
exercise_to_concept_id = {
    name: concept_to_id[exercise_to_concept.get(name, name)]
    for name in question_names
}

NUM_QUESTIONS = len(question_to_id)
NUM_CONCEPTS = len(concept_to_id)

print('NUM_QUESTIONS =', NUM_QUESTIONS)
print('NUM_CONCEPTS =', NUM_CONCEPTS)

NUM_QUESTIONS = 722
NUM_CONCEPTS = 40


## 第二遍扫描：构建全量数值化交互表

为了降低内存占用，这一步只保留训练需要的数值列：

- `user_id`
- `timestamp`
- `question_ids`
- `concept_ids`
- `concept_ids_internal`
- `correct`
- `count_attempts`
- `count_hints`
- `time_taken`

In [7]:
load_start = time.time()
usecols = ['user_id', 'exercise', 'time_done', 'time_taken', 'correct', 'count_attempts', 'count_hints']
dtypes = {
    'user_id': 'int32',
    'exercise': 'string',
    'time_done': 'int64',
    'time_taken': 'float32',
    'correct': 'boolean',
    'count_attempts': 'float32',
    'count_hints': 'float32',
}

frames = []
kept_rows = 0
for chunk_idx, chunk in enumerate(pd.read_csv(
    RAW_JUNYI_PATH,
    usecols=usecols,
    dtype=dtypes,
    chunksize=CHUNKSIZE,
    low_memory=False,
)):
    if chosen_user_set:
        chunk = chunk[chunk['user_id'].isin(chosen_user_set)]
    if chunk.empty:
        continue

    exercise_series = chunk['exercise'].astype(str)
    question_ids = exercise_series.map(question_to_id)
    concept_ids = exercise_series.map(exercise_to_concept_id)
    valid_mask = question_ids.notna() & concept_ids.notna() & chunk['time_done'].notna()
    chunk = chunk.loc[valid_mask]
    question_ids = question_ids.loc[valid_mask].astype(np.int32)
    concept_ids = concept_ids.loc[valid_mask].astype(np.int32)

    numeric_chunk = pd.DataFrame({
        'user_id': chunk['user_id'].to_numpy(dtype=np.int32),
        'timestamp': chunk['time_done'].to_numpy(dtype=np.int64),
        'question_ids': question_ids.to_numpy(dtype=np.int32),
        'concept_ids': concept_ids.to_numpy(dtype=np.int32),
        'concept_ids_internal': (concept_ids.to_numpy(dtype=np.int32) - 1).astype(np.int32),
        'correct': chunk['correct'].fillna(False).to_numpy(dtype=np.int8),
        'count_attempts': pd.to_numeric(chunk['count_attempts'], errors='coerce').fillna(1.0).clip(lower=1.0).to_numpy(dtype=np.float32),
        'count_hints': pd.to_numeric(chunk['count_hints'], errors='coerce').fillna(0.0).clip(lower=0.0).to_numpy(dtype=np.float32),
        'time_taken': pd.to_numeric(chunk['time_taken'], errors='coerce').replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32),
    })
    frames.append(numeric_chunk)
    kept_rows += len(numeric_chunk)

    if (chunk_idx + 1) % 5 == 0:
        print(f'已读取 chunk {chunk_idx + 1}, kept_rows={kept_rows:,}')

df = pd.concat(frames, ignore_index=True)
del frames
gc.collect()

median_time = float(pd.Series(df['time_taken']).dropna().median())
if not np.isfinite(median_time) or median_time <= 0:
    median_time = 10.0
df['time_taken'] = pd.Series(df['time_taken']).fillna(median_time).clip(lower=1.0).astype(np.float32)

df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

print('过滤后交互数 =', f'{len(df):,}')
print('过滤后用户数 =', f"{df['user_id'].nunique():,}")
print('过滤后 question 数 =', f"{df['question_ids'].nunique():,}")
print('过滤后 concept 数 =', f"{df['concept_ids'].nunique():,}")
print('median_time =', median_time)
print('读取与数值化耗时(秒) =', round(time.time() - load_start, 2))
df.head()

已读取 chunk 5, kept_rows=143,313
已读取 chunk 10, kept_rows=286,713
已读取 chunk 15, kept_rows=430,131
已读取 chunk 20, kept_rows=573,829
已读取 chunk 25, kept_rows=717,110
过滤后交互数 = 743,924
过滤后用户数 = 5,000
过滤后 question 数 = 684
过滤后 concept 数 = 39
median_time = 8.0
读取与数值化耗时(秒) = 39.01


,user_id,timestamp,question_ids,concept_ids,concept_ids_internal,correct,count_attempts,count_hints,time_taken
0,348,1420700023585240,427,2,1,1,1.0,0.0,4.0
1,348,1420700028821400,427,2,1,1,1.0,0.0,3.0
2,348,1420700035674120,427,2,1,1,1.0,0.0,5.0
3,348,1420700039827800,427,2,1,1,1.0,0.0,3.0
4,348,1420700044833640,427,2,1,1,1.0,0.0,3.0


## 计算训练集难度

难度只用训练用户计算，避免测试集信息泄露。

这里沿用 quickstart 的简单离散方案：

```text
difficulty = floor((1 - train_correct_rate) * 10) + 1
```

范围裁剪到 `1..10`。

In [8]:
train_mask = df['user_id'].isin(train_user_ids)
train_df = df.loc[train_mask]
global_correct = float(train_df['correct'].mean())
default_difficulty = int(np.clip(np.floor((1.0 - global_correct) * 10) + 1, 1, 10))

question_acc = train_df.groupby('question_ids', sort=False)['correct'].mean()
concept_acc = train_df.groupby('concept_ids', sort=False)['correct'].mean()

question_difficulty_lookup = np.full(NUM_QUESTIONS + 1, default_difficulty, dtype=np.int32)
concept_difficulty_lookup = np.full(NUM_CONCEPTS + 1, default_difficulty, dtype=np.int32)
question_difficulty_lookup[question_acc.index.to_numpy(dtype=np.int32)] = np.clip(
    np.floor((1.0 - question_acc.to_numpy()) * 10).astype(np.int32) + 1,
    1,
    10,
)
concept_difficulty_lookup[concept_acc.index.to_numpy(dtype=np.int32)] = np.clip(
    np.floor((1.0 - concept_acc.to_numpy()) * 10).astype(np.int32) + 1,
    1,
    10,
)

df['question_difficulty'] = question_difficulty_lookup[df['question_ids'].to_numpy(dtype=np.int32)]
df['concept_difficulty'] = concept_difficulty_lookup[df['concept_ids'].to_numpy(dtype=np.int32)]

print('训练集全局正确率 =', round(global_correct, 6))
print('默认难度桶 =', default_difficulty)
print(df[['question_ids', 'concept_ids', 'question_difficulty', 'concept_difficulty']].head())

训练集全局正确率 = 0.829236
默认难度桶 = 2
   question_ids  concept_ids  question_difficulty  concept_difficulty
0           427            2                    1                   1
1           427            2                    1                   1
2           427            2                    1                   1
3           427            2                    1                   1
4           427            2                    1                   1


## 拟合 AHS 行为聚类

AHS 特征：

- `attempts = log1p(clipped count_attempts)`
- `hints = log1p(clipped count_hints)`
- `speed = log1p(clipped 60 / time_taken)`

聚类只在训练用户上拟合，然后应用到全量交互。

In [9]:
def compute_behavior_features(frame, clip_values=None):
    attempts_raw = frame['count_attempts'].to_numpy(dtype=np.float32)
    hints_raw = frame['count_hints'].to_numpy(dtype=np.float32)
    speed_raw = (60.0 / np.maximum(frame['time_taken'].to_numpy(dtype=np.float32), 1.0)).astype(np.float32)

    if clip_values is None:
        clip_values = {
            'attempts': float(np.quantile(attempts_raw, 0.99)),
            'hints': float(np.quantile(hints_raw, 0.99)),
            'speed': float(np.quantile(speed_raw, 0.99)),
        }

    attempts = np.log1p(np.clip(attempts_raw, 0.0, clip_values['attempts'])).astype(np.float32)
    hints = np.log1p(np.clip(hints_raw, 0.0, clip_values['hints'])).astype(np.float32)
    speed = np.log1p(np.clip(speed_raw, 0.0, clip_values['speed'])).astype(np.float32)
    return attempts, hints, speed, np.stack([attempts, hints, speed], axis=-1), clip_values

train_attempts, train_hints, train_speed, train_features, clip_values = compute_behavior_features(train_df)
scaler = StandardScaler()
scaled_train_features = scaler.fit_transform(train_features)

if CLUSTER_SAMPLE_SIZE and len(scaled_train_features) > CLUSTER_SAMPLE_SIZE:
    sample_indices = np.random.default_rng(SEED).choice(len(scaled_train_features), size=CLUSTER_SAMPLE_SIZE, replace=False)
    fit_features = scaled_train_features[sample_indices]
else:
    fit_features = scaled_train_features

cluster_model = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=SEED,
    n_init=20,
    batch_size=4096,
)
cluster_model.fit(fit_features)
raw_centers = scaler.inverse_transform(cluster_model.cluster_centers_)
cluster_order = sorted(
    range(len(raw_centers)),
    key=lambda i: (float(raw_centers[i, 0]), float(raw_centers[i, 1]), float(-raw_centers[i, 2])),
)
cluster_mapping = {int(old_label): int(new_label + 1) for new_label, old_label in enumerate(cluster_order)}
behavior_centers = []
for old_label, center in enumerate(raw_centers):
    cluster_id = cluster_mapping[int(old_label)]
    behavior_centers.append({
        'cluster_id': int(cluster_id),
        'center_attempts_log': float(center[0]),
        'center_hints_log': float(center[1]),
        'center_speed_log': float(center[2]),
        'center_attempts_raw': float(np.expm1(center[0])),
        'center_hints_raw': float(np.expm1(center[1])),
        'center_speed_qpm_raw': float(np.expm1(center[2])),
    })
behavior_centers = sorted(behavior_centers, key=lambda item: item['cluster_id'])

all_attempts, all_hints, all_speed, all_features, _ = compute_behavior_features(df, clip_values=clip_values)
raw_labels = cluster_model.predict(scaler.transform(all_features))
df['attempts'] = all_attempts
df['hints'] = all_hints
df['speed'] = all_speed
df['behavior_cluster'] = np.array([cluster_mapping[int(label)] for label in raw_labels], dtype=np.int32)

cluster_counts = df['behavior_cluster'].value_counts().sort_index()
print('clip_values =', clip_values)
print('behavior_centers =')
print(json.dumps(behavior_centers, ensure_ascii=False, indent=2))
print('cluster_counts =')
print(cluster_counts)

del train_features, scaled_train_features, all_features, raw_labels
gc.collect()

OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.


clip_values = {'attempts': 6.0, 'hints': 7.0, 'speed': 60.0}
behavior_centers =
[
  {
    "cluster_id": 1,
    "center_attempts_log": 0.6931425333023071,
    "center_hints_log": 0.004017433617264032,
    "center_speed_log": 1.4475970268249512,
    "center_attempts_raw": 0.999990701675415,
    "center_hints_raw": 0.0040255142375826836,
    "center_speed_qpm_raw": 3.252882719039917
  },
  {
    "cluster_id": 2,
    "center_attempts_log": 0.6966875195503235,
    "center_hints_log": 0.002341261599212885,
    "center_speed_log": 2.802048444747925,
    "center_attempts_raw": 1.0070931911468506,
    "center_hints_raw": 0.002344004577025771,
    "center_speed_qpm_raw": 15.47836685180664
  },
  {
    "cluster_id": 3,
    "center_attempts_log": 1.0720124244689941,
    "center_hints_log": 1.6647273302078247,
    "center_speed_log": 1.156477451324463,
    "center_attempts_raw": 1.9212523698806763,
    "center_hints_raw": 4.284232139587402,
    "center_speed_qpm_raw": 2.1787164211273193
  },
  {
  

33

## 保存全量 `.npz` 序列包

这里分 split 构建、保存、释放内存，避免同时持有三个 split 的大 bundle。

In [10]:
def build_save_bundle_for_split(split_name, user_ids, output_path, shuffle=False):
    start = time.time()
    records = build_sequence_records(df, user_ids, SEQUENCE_LENGTH)
    if shuffle:
        records = shuffle_records(records, seed=2)
    bundle = records_to_bundle(records, sequence_length=SEQUENCE_LENGTH)
    save_bundle_npz(bundle, output_path)
    summary = {
        'split': split_name,
        'users': int(len(user_ids)),
        'sequences': int(bundle.num_samples),
        'sequence_length': int(bundle.sequence_length),
        'output_path': str(output_path),
        'elapsed_seconds': float(time.time() - start),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    del records, bundle
    gc.collect()
    return summary

train_summary = build_save_bundle_for_split('train', train_user_ids, TRAIN_NPZ_PATH, shuffle=True)
valid_summary = build_save_bundle_for_split('valid', valid_user_ids, VALID_NPZ_PATH, shuffle=True)
test_summary = build_save_bundle_for_split('test', test_user_ids, TEST_NPZ_PATH, shuffle=False)

{
  "split": "train",
  "users": 4000,
  "sequences": 8720,
  "sequence_length": 100,
  "output_path": "/root/autodl-tmp/ahs-kt/data/junyi/junyi_full_train_ahskt.npz",
  "elapsed_seconds": 4.113116502761841
}
{
  "split": "valid",
  "users": 500,
  "sequences": 1024,
  "sequence_length": 100,
  "output_path": "/root/autodl-tmp/ahs-kt/data/junyi/junyi_full_valid_ahskt.npz",
  "elapsed_seconds": 0.3912789821624756
}
{
  "split": "test",
  "users": 500,
  "sequences": 1190,
  "sequence_length": 100,
  "output_path": "/root/autodl-tmp/ahs-kt/data/junyi/junyi_full_test_ahskt.npz",
  "elapsed_seconds": 0.40546154975891113
}


In [11]:
metadata = {
    'dataset_name': 'junyi',
    'subset_mode': 'all_eligible_users' if MAX_USERS is None else 'sampled_users',
    'max_users': None if MAX_USERS is None else int(MAX_USERS),
    'min_interactions': int(MIN_INTERACTIONS),
    'sequence_length': int(SEQUENCE_LENGTH),
    'num_questions': int(NUM_QUESTIONS),
    'num_concepts': int(NUM_CONCEPTS),
    'num_question_difficulty': 10,
    'num_concept_difficulty': 10,
    'num_behavior_clusters': int(N_CLUSTERS + 1),
    'num_raw_interactions': int(total_rows),
    'num_filtered_interactions': int(len(df)),
    'num_users': int(df['user_id'].nunique()),
    'global_correct_train': float(global_correct),
    'default_difficulty': int(default_difficulty),
    'clip_values': {key: float(value) for key, value in clip_values.items()},
    'behavior_centers': behavior_centers,
    'cluster_counts': {str(int(k)): int(v) for k, v in cluster_counts.items()},
    'split_summary': {
        'train': train_summary,
        'valid': valid_summary,
        'test': test_summary,
    },
}
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved metadata:', METADATA_PATH)
print(json.dumps(metadata['split_summary'], ensure_ascii=False, indent=2))

saved metadata: /root/autodl-tmp/ahs-kt/data/junyi/junyi_full_metadata.json
{
  "train": {
    "split": "train",
    "users": 4000,
    "sequences": 8720,
    "sequence_length": 100,
    "output_path": "/root/autodl-tmp/ahs-kt/data/junyi/junyi_full_train_ahskt.npz",
    "elapsed_seconds": 4.113116502761841
  },
  "valid": {
    "split": "valid",
    "users": 500,
    "sequences": 1024,
    "sequence_length": 100,
    "output_path": "/root/autodl-tmp/ahs-kt/data/junyi/junyi_full_valid_ahskt.npz",
    "elapsed_seconds": 0.3912789821624756
  },
  "test": {
    "split": "test",
    "users": 500,
    "sequences": 1190,
    "sequence_length": 100,
    "output_path": "/root/autodl-tmp/ahs-kt/data/junyi/junyi_full_test_ahskt.npz",
    "elapsed_seconds": 0.40546154975891113
  }
}


## 生成 full 版 config

In [12]:
config_payload = {
    'project_name': 'ahs-kt',
    'task_name': TASK_NAME,
    'seed': int(SEED),
    'dataset': {
        'mode': 'real_npz',
        'train_path': str(TRAIN_NPZ_PATH.relative_to(PROJECT_ROOT)),
        'valid_path': str(VALID_NPZ_PATH.relative_to(PROJECT_ROOT)),
        'test_path': str(TEST_NPZ_PATH.relative_to(PROJECT_ROOT)),
    },
    'model': {
        'num_questions': int(NUM_QUESTIONS),
        'num_concepts': int(NUM_CONCEPTS),
        'num_question_difficulty': 10,
        'num_concept_difficulty': 10,
        'num_behavior_clusters': int(N_CLUSTERS + 1),
        'sequence_length': int(SEQUENCE_LENGTH),
        'embedding_dim': 64,
        'difficulty_dim': 32,
        'behavior_dim': 32,
        'hidden_dim': 96,
        'dropout': 0.2,
        'use_behavior_cluster': True,
        'use_difficulty_features': True,
        'use_behavior_features': True,
        'use_target_interaction': False,
        'fusion_mode': 'early',
        'behavior_condition_on_difficulty': True,
        'aux_residual_scale': 0.25,
        'difficulty_mode': 'embedding',
        'difficulty_bias_scale': 0.1,
        'difficulty_feature_source': 'question_concept',
        'question_global_easiness': float(global_correct),
        'concept_global_easiness': float(global_correct),
    },
    'training': {
        'epochs': int(EPOCHS),
        'batch_size': int(BATCH_SIZE),
        'learning_rate': float(LEARNING_RATE),
        'patience': int(PATIENCE),
    },
    'demo': {
        'train_size': 0,
        'valid_size': 0,
        'test_size': 0,
    },
    'outputs': {
        'root_dir': str(OUTPUT_DIR.relative_to(PROJECT_ROOT)),
    },
}
CONFIG_PATH.write_text(json.dumps(config_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved config:', CONFIG_PATH)
print(json.dumps(config_payload, ensure_ascii=False, indent=2)[:3000])

saved config: /root/autodl-tmp/ahs-kt/configs/ahskt_junyi_full.json
{
  "project_name": "ahs-kt",
  "task_name": "ahskt_junyi_full",
  "seed": 2026,
  "dataset": {
    "mode": "real_npz",
    "train_path": "data/junyi/junyi_full_train_ahskt.npz",
    "valid_path": "data/junyi/junyi_full_valid_ahskt.npz",
    "test_path": "data/junyi/junyi_full_test_ahskt.npz"
  },
  "model": {
    "num_questions": 722,
    "num_concepts": 40,
    "num_question_difficulty": 10,
    "num_concept_difficulty": 10,
    "num_behavior_clusters": 5,
    "sequence_length": 100,
    "embedding_dim": 64,
    "difficulty_dim": 32,
    "behavior_dim": 32,
    "hidden_dim": 96,
    "dropout": 0.2,
    "use_behavior_cluster": true,
    "use_difficulty_features": true,
    "use_behavior_features": true,
    "use_target_interaction": false,
    "fusion_mode": "early",
    "behavior_condition_on_difficulty": true,
    "aux_residual_scale": 0.25,
    "difficulty_mode": "embedding",
    "difficulty_bias_scale": 0.1,
    "

## 训练与评估

这一段会加载刚才保存的三个 `.npz`，然后调用项目原生训练流程。

如果这里内存不够，先把 `BATCH_SIZE` 调小；如果仍然不够，需要把项目的数据管线改成流式 `TFRecord` 或 `memmap`，而不是一次性 `from_tensor_slices`。

In [13]:
config = load_config(CONFIG_PATH, project_root=PROJECT_ROOT)
tf.random.set_seed(config.seed)

load_start = time.time()
train_bundle_loaded, valid_bundle_loaded, test_bundle_loaded = load_bundle_from_config(config)
print('加载 bundle 耗时(秒) =', round(time.time() - load_start, 2))
print('train / valid / test sequences =', train_bundle_loaded.num_samples, valid_bundle_loaded.num_samples, test_bundle_loaded.num_samples)

model = AHSKTModel(config.model)
train_start = time.time()
metrics_summary = fit_and_evaluate(
    model=model,
    train_bundle=train_bundle_loaded,
    valid_bundle=valid_bundle_loaded,
    test_bundle=test_bundle_loaded,
    config=config,
)
train_elapsed = time.time() - train_start

metrics_path = config.output_root / f'{config.task_name}_metrics.json'
metrics_path.write_text(json.dumps(metrics_summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('训练耗时(秒) =', round(train_elapsed, 2))
print('metrics_path =', metrics_path)
print(json.dumps(metrics_summary['test_metrics'], ensure_ascii=False, indent=2))

加载 bundle 耗时(秒) = 0.36
train / valid / test sequences = 8720 1024 1190
训练耗时(秒) = 26.03
metrics_path = /root/autodl-tmp/ahs-kt/outputs/junyi_full_run/ahskt_junyi_full_metrics.json
{
  "loss": 0.3971349895000458,
  "auc": 0.7530843730725019,
  "acc": 0.8412622117545272,
  "rmse": 0.3493794798851013
}


## 额外计算 F1

项目原生 metrics 默认输出 `loss / auc / acc / rmse`。这里额外基于测试集预测计算 `F1`。

In [14]:
def collect_targets_and_predictions(model, bundle, batch_size):
    dataset = bundle.to_tf_dataset(batch_size=batch_size, shuffle=False)
    all_targets = []
    all_predictions = []
    for batch in dataset:
        logits = model(batch, training=False)
        next_logits = logits[:, :-1]
        next_targets = tf.cast(batch['responses'][:, 1:], tf.float32)
        next_mask = tf.cast(batch['mask'][:, 1:], tf.float32)
        valid_logits = tf.boolean_mask(next_logits, next_mask > 0)
        valid_targets = tf.boolean_mask(next_targets, next_mask > 0)
        all_targets.append(valid_targets.numpy())
        all_predictions.append(tf.sigmoid(valid_logits).numpy())
    return np.concatenate(all_targets, axis=0), np.concatenate(all_predictions, axis=0)

test_targets, test_predictions = collect_targets_and_predictions(
    model=model,
    bundle=test_bundle_loaded,
    batch_size=config.training.batch_size,
)

test_binary_predictions = (test_predictions > 0.5).astype(int)
summary_with_f1 = {
    'acc': float(accuracy_score(test_targets, test_binary_predictions)),
    'auc': float(roc_auc_score(test_targets, test_predictions)),
    'f1': float(f1_score(test_targets, test_binary_predictions)),
    'loss': float(metrics_summary['test_metrics']['loss']),
    'rmse': float(metrics_summary['test_metrics']['rmse']),
    'num_test_points': int(len(test_targets)),
    'best_epoch': int(metrics_summary['best_epoch']),
    'best_valid_auc': float(metrics_summary['best_valid_auc']),
}

summary_with_f1_path = config.output_root / f'{config.task_name}_metrics_with_f1.json'
summary_with_f1_path.write_text(json.dumps(summary_with_f1, ensure_ascii=False, indent=2), encoding='utf-8')

print('summary_with_f1_path =', summary_with_f1_path)
print(json.dumps(summary_with_f1, ensure_ascii=False, indent=2))

summary_with_f1_path = /root/autodl-tmp/ahs-kt/outputs/junyi_full_run/ahskt_junyi_full_metrics_with_f1.json
{
  "acc": 0.8412622117545272,
  "auc": 0.7530843730725019,
  "f1": 0.9104090933752916,
  "loss": 0.3971349895000458,
  "rmse": 0.3493794798851013,
  "num_test_points": 83219,
  "best_epoch": 10,
  "best_valid_auc": 0.7317489777254513
}


## 输出文件位置

执行完成后，关键文件是：

- `data/junyi/junyi_full_train_ahskt.npz`
- `data/junyi/junyi_full_valid_ahskt.npz`
- `data/junyi/junyi_full_test_ahskt.npz`
- `data/junyi/junyi_full_metadata.json`
- `configs/ahskt_junyi_full.json`
- `outputs/junyi_full_run/ahskt_junyi_full_metrics.json`
- `outputs/junyi_full_run/ahskt_junyi_full_metrics_with_f1.json`

如果要先小规模试跑，把参数区的 `MAX_USERS = None` 改成 `MAX_USERS = 5000` 或更小即可。